# 24b7 — SDP Campaign Performance Validation v2 Fixed + 2026 Nullable Boolean Fix

This notebook fixes the issues found in `24b_sdp_campaign_performance_validation_v2.ipynb`.

Main fixes:

1. Correctly recognises both `SDP` and `Social Democratic Party` labels.
2. Treats `sdp_campaign_wards_profile_v1.csv` as an already-filtered SDP dataset, rather than refiltering it incorrectly.
3. Calculates effective SDP vote share where possible.
4. Attempts improved WD25 matching using:
   - existing WD25 codes;
   - 2023/other source-year geography crosswalks where lookup files exist;
   - result-area key joins to `ward_result_summary_v2_with_2023_matches.csv` where available;
   - conservative name fallback.
5. Writes the same v2 output filenames expected by Notebook 24c.

Run this notebook **instead of** the previous 24b, then run Notebook 24c.

Additional 2026 import fix in this version:

- Accepts the 2026 provisional CSV headers such as `Election Year`, `WD26NM`, `LAD26NM`, `Candidate`, `Votes`, `Votes (Perc.)` and `Turnout`.
- Coalesces aliases row-by-row after concatenating files, so historical snake_case columns and 2026 title-case columns can coexist.
- Parses percentage strings such as `2.60%` correctly.

Additional 24b7 fix:

- Avoids `TypeError: boolean value of NA is ambiguous` when parsing vote-share percentages from nullable pandas columns.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"
GEOGRAPHY_DIR = DATA_DIR / "geography"

INPUT_DIRS = [
    PROCESSED_DIR / "sdp_campaign_validation_v1",
    PROCESSED_DIR / "election_results",
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR / "caveat_resolution_v2",
    GEOGRAPHY_DIR,
    DATA_DIR,
    PROJECT_DIR,
]

OUTPUT_DIR = PROCESSED_DIR / "sdp_campaign_validation_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs. Edit these if your filenames differ.
EXISTING_SDP_PROFILE_FILENAME = "sdp_campaign_wards_profile_v1.csv"
LOCAL_ELECTION_RAW_FILENAME = "local_election_results_raw_v1.csv"
WARD_SUMMARY_MATCHED_FILENAME = "ward_result_summary_v2_with_2023_matches.csv"
PROVISIONAL_2026_FILENAME = "sdp_candidate_results_2026_raw_v1.csv"
SDP_WORKBOOK_FILENAME = "SDP Election Results since 2018.xlsx"
MODEL_FILENAME_CANDIDATES = [
    "all_available_consolidated_target_review_v1.csv",
    "north_west_revised_consolidated_review_v2.csv",
    "target_score_components_ward25_all_available_v2.csv",
]

# Behaviour controls.
USE_EXISTING_PROFILE_IF_AVAILABLE = True
INCLUDE_RAW_EXTRA_IF_EXISTING_PROFILE_AVAILABLE = False  # Prevents duplicate 2021-2025 rows.
SOURCE_TO_WD25_MIN_SHARE = 0.50

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v2


In [2]:
def find_file(filename, required=False):
    for folder in INPUT_DIRS + [Path("/mnt/data")]:
        p = folder / filename
        if p.exists():
            return p
    for root in [PROCESSED_DIR, RAW_DIR, GEOGRAPHY_DIR, DATA_DIR, PROJECT_DIR]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(filename)
    return None


def read_csv_optional(filename):
    p = find_file(filename, required=False)
    if p is None:
        print("Optional CSV missing:", filename)
        return None, None
    df = pd.read_csv(p, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {p}")
    return df, p


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip().replace("&", " and ")
    x = re.sub(r"\bst[.]?\b", "saint", x)
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def clean_colname(x):
    """Normalise column names so title-case/spaced headers and snake_case headers can both be matched."""
    if x is None:
        return ""
    x = str(x).strip().lower().replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", "_", x)
    return re.sub(r"_+", "_", x).strip("_")


def norm_code(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x if x and x.lower() not in ["nan", "none", "null", "<na>"] else np.nan


def to_num(s):
    """Numeric parser that handles commas and percentage signs such as '2.60%'."""
    if isinstance(s, pd.Series):
        cleaned = (
            s.astype("string")
             .str.replace(",", "", regex=False)
             .str.replace("%", "", regex=False)
             .str.strip()
        )
        return pd.to_numeric(cleaned, errors="coerce")
    if pd.isna(s):
        return np.nan
    return pd.to_numeric(str(s).replace(",", "").replace("%", "").strip(), errors="coerce")


def is_sdp_party(series):
    # Critical fix: use actual regex word boundary, not a backspace character.
    s = series.fillna("").astype(str).str.lower()
    positive = s.str.contains(r"\bsdp\b|social democratic party", regex=True, na=False)
    negative = s.str.contains(r"sdlp|social democratic labour", regex=True, na=False)
    return positive & ~negative


def matching_columns(df, candidates):
    """Return all columns matching candidate names, using exact and normalised header comparison."""
    out = []
    existing = list(df.columns)
    normalised_to_original = {}
    for col in existing:
        normalised_to_original.setdefault(clean_colname(col), []).append(col)

    for candidate in candidates:
        # Exact match first.
        if candidate in df.columns and candidate not in out:
            out.append(candidate)

        # Normalised match second.
        key = clean_colname(candidate)
        for col in normalised_to_original.get(key, []):
            if col not in out:
                out.append(col)
    return out


def first_col(df, candidates):
    cols = matching_columns(df, candidates)
    return cols[0] if cols else None


def combine_first_cols(df, candidates):
    """Combine all matching alias columns row-wise.

    This matters when concatenating historical profile rows and the 2026 provisional file:
    older rows may use snake_case headers while the 2026 file uses headers like 'Election Year',
    'WD26NM', 'Votes (Perc.)', etc.
    """
    cols = matching_columns(df, candidates)
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    for col in cols:
        out = out.combine_first(df[col])
    return out


## 24b2.1 Load model / atlas table

This is the WD25 model table used for joining SDP campaign rows to the tribe and score model.

In [3]:
model = None
model_path = None
for fname in MODEL_FILENAME_CANDIDATES:
    p = find_file(fname, required=False)
    if p is not None:
        model = pd.read_csv(p, low_memory=False)
        model_path = p
        break

if model is None:
    raise FileNotFoundError("No model/review file found. Provide all_available_consolidated_target_review_v1.csv or equivalent.")

model["WD25CD"] = model["WD25CD"].map(norm_code)
model_lookup = model.dropna(subset=["WD25CD"]).drop_duplicates("WD25CD").copy()
model_codes = set(model_lookup["WD25CD"])
print("Loaded model:", model.shape, "from", model_path)
print("Unique WD25 codes:", len(model_codes))

Loaded model: (7572, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\all_available_consolidated_target_review_v1.csv
Unique WD25 codes: 7572


## 24b2.2 Load SDP candidate/result rows

If the existing v1 SDP profile exists, it is treated as an already-filtered SDP dataset. This avoids the previous bug where rows labelled simply `SDP` were dropped.

In [4]:
frames = []
source_notes = []

existing, existing_path = read_csv_optional(EXISTING_SDP_PROFILE_FILENAME)
if existing is not None and USE_EXISTING_PROFILE_IF_AVAILABLE:
    existing = existing.copy()
    existing["source_mode"] = "existing_sdp_profile_v1"
    frames.append(existing)
    source_notes.append(("existing_sdp_profile_v1", len(existing), str(existing_path)))

raw, raw_path = read_csv_optional(LOCAL_ELECTION_RAW_FILENAME)
if raw is not None and "party_label" in raw.columns:
    if existing is None or INCLUDE_RAW_EXTRA_IF_EXISTING_PROFILE_AVAILABLE:
        sdp_raw = raw[is_sdp_party(raw["party_label"])].copy()
        sdp_raw["source_mode"] = "local_election_results_raw_extracted"
        frames.append(sdp_raw)
        source_notes.append(("local_election_results_raw_extracted", len(sdp_raw), str(raw_path)))
    else:
        print("Raw election file found, but skipped to avoid duplicating existing v1 profile rows.")

prov26, prov26_path = read_csv_optional(PROVISIONAL_2026_FILENAME)
if prov26 is not None:
    prov26 = prov26.copy()
    prov26["source_mode"] = "provisional_2026_sdp_file"
    frames.append(prov26)
    source_notes.append(("provisional_2026_sdp_file", len(prov26), str(prov26_path)))

# Optional workbook import, only if no existing profile and no raw extraction.
workbook_path = find_file(SDP_WORKBOOK_FILENAME, required=False)
if workbook_path is not None and existing is None:
    try:
        xls = pd.ExcelFile(workbook_path)
        wb_frames = []
        for sheet in xls.sheet_names:
            temp = pd.read_excel(workbook_path, sheet_name=sheet)
            if len(temp) == 0: continue
            temp.columns = [clean_text(c) for c in temp.columns]
            if any("candidate" in c for c in temp.columns) or any("votes" in c for c in temp.columns) or any("sdp" in c for c in temp.columns):
                temp["source_sheet"] = sheet
                temp["source_mode"] = "sdp_workbook_import"
                wb_frames.append(temp)
        if wb_frames:
            wb = pd.concat(wb_frames, ignore_index=True, sort=False)
            frames.append(wb)
            source_notes.append(("sdp_workbook_import", len(wb), str(workbook_path)))
    except Exception as e:
        print("Workbook import failed; continuing:", e)

if not frames:
    raise FileNotFoundError("No SDP candidate/result data found.")

sdp_raw_combined = pd.concat(frames, ignore_index=True, sort=False)
print("Combined SDP candidate/result rows before standardisation:", len(sdp_raw_combined))
display(pd.DataFrame(source_notes, columns=["source", "rows", "path"]))

Loaded sdp_campaign_wards_profile_v1.csv: (171, 76) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_campaign_wards_profile_v1.csv
Loaded local_election_results_raw_v1.csv: (80392, 52) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\local_election_results_raw_v1.csv
Loaded sdp_candidate_results_2026_raw_v1.csv: (48, 30) from c:\Users\keena\Documents\Electoral_Tribes\data\raw\sdp_candidate_results_2026_raw_v1.csv
Combined SDP candidate/result rows before standardisation: 219


,source,rows,path
0,existing_sdp_profile_v1,171,c:\Users\keena\Documents\Electoral_Tribes\data...
1,provisional_2026_sdp_file,48,c:\Users\keena\Documents\Electoral_Tribes\data...


## 24b2.3 Standardise columns and calculate effective SDP vote share

In [5]:
aliases = {
    "election_year": ["election_year", "source_year", "year", "Election Year"],
    "election_date": ["election_date", "date", "Election Date"],
    "council_name": ["council_name", "council", "local_authority", "lad_name", "LAD25NM", "LAD26NM", "Authority", "Local Authority"],
    "ward_name": ["ward_name", "ward", "division_name", "WD25NM", "WD26NM", "Ward", "Division"],
    "candidate_name": ["candidate_name", "candidate", "name", "Candidate"],
    "party_label": ["party_label", "party", "description", "standard_party_label", "Party"],
    "sdp_votes": ["sdp_votes", "votes", "Votes", "votes_effective", "candidate_votes"],
    "valid_votes": ["valid_votes", "Valid Votes", "Turnout", "candidate_effective_total_votes", "total_votes"],
    "vote_share": ["sdp_vote_share", "vote_share", "share", "Votes (Perc.)", "Vote Share"],
    "sdp_position": ["sdp_position", "rank", "position", "Place"],
    "winner_party": ["winner_party", "top_party_by_votes", "top_party_bucket", "Previous Winner"],
    "runner_up_party": ["runner_up_party", "runner_up_party_by_votes", "runner_up_party_bucket"],

    # The validation model is still a WD25 atlas, but 2026 result files may carry WD26/LAD26 headers.
    # These are first normalised into the model-facing WD25/LAD25 working fields; rows are then matched
    # by code where possible, or by LAD+ward name fallback where codes are missing.
    "WD25CD": ["WD25CD", "WD26CD", "matched_wd25cd", "matched_wd26cd", "ward25cd", "ward26cd", "wd25cd", "wd26cd"],
    "WD25NM": ["WD25NM", "WD26NM", "matched_wd25nm", "matched_wd26nm", "ward25nm", "ward26nm", "wd25nm", "wd26nm"],
    "LAD25CD": ["LAD25CD", "LAD26CD", "matched_lad25cd", "matched_lad26cd", "lad25cd", "lad26cd"],
    "LAD25NM": ["LAD25NM", "LAD26NM", "matched_lad25nm", "matched_lad26nm", "lad25nm", "lad26nm"],

    "ward_code": ["ward_code", "source_geography_code", "ec_ward_code", "WD26CD", "WD25CD"],
    "result_area_key": ["result_area_key"],
    "boundary_year": ["boundary_year", "source_boundary_year"],
}

std = pd.DataFrame(index=sdp_raw_combined.index)
for standard, candidates in aliases.items():
    # Use row-wise alias coalescing rather than one global first column.
    # This fixes mixed input frames, especially historical snake_case rows + 2026 title-case rows.
    std[standard] = combine_first_cols(sdp_raw_combined, candidates)

for c in ["source_mode", "source_file", "source_sheet", "source_url", "source_notes", "mapping_confidence", "mapping_notes", "result_id"]:
    std[c] = sdp_raw_combined[c] if c in sdp_raw_combined.columns else np.nan

# Keep SDP rows. Existing profile and provisional 2026 are trusted as already SDP-filtered.
trusted_modes = ["existing_sdp_profile_v1", "provisional_2026_sdp_file"]
party_s = std["party_label"].fillna("").astype(str)
keep = is_sdp_party(party_s) | std["source_mode"].isin(trusted_modes)
std = std[keep].copy()

std["election_year"] = to_num(std["election_year"]).astype("Int64")
std["source_year"] = std["election_year"]
std["boundary_year"] = to_num(std["boundary_year"]).fillna(std["source_year"]).astype("Int64")
std["sdp_votes"] = to_num(std["sdp_votes"])
std["valid_votes"] = to_num(std["valid_votes"])

vote_share_text = std["vote_share"].astype("string")
raw_share = pd.Series(to_num(std["vote_share"]), index=std.index, dtype="float64")

# If the source text contains a percent sign, always convert percent points to proportions.
# This fixes cases such as "0.20%" which should become 0.002, not 0.20.
# Use pandas masks rather than np.where because nullable pandas booleans can contain pd.NA.
has_percent_sign = (
    vote_share_text
    .str.contains("%", regex=False, na=False)
    .fillna(False)
    .astype(bool)
)
needs_percent_scaling = has_percent_sign | raw_share.gt(1).fillna(False)
std["sdp_vote_share_raw_clean"] = raw_share.mask(needs_percent_scaling, raw_share / 100)

# Calculate vote share from votes only where valid_votes is known and greater than zero.
# This avoids the same pd.NA boolean issue that can occur inside np.where.
has_valid_votes = std["valid_votes"].gt(0).fillna(False)
std["sdp_vote_share_from_votes"] = np.nan
std.loc[has_valid_votes, "sdp_vote_share_from_votes"] = (
    std.loc[has_valid_votes, "sdp_votes"] / std.loc[has_valid_votes, "valid_votes"]
)
std["sdp_vote_share_effective"] = std["sdp_vote_share_raw_clean"].combine_first(std["sdp_vote_share_from_votes"])

for c in ["WD25CD", "ward_code"]:
    std[c] = std[c].map(norm_code)
std["clean_council_name"] = std["council_name"].map(clean_text)
std["clean_ward_name"] = std["ward_name"].map(clean_text)

# De-duplicate conservatively.
dedupe_cols = ["election_year", "council_name", "ward_name", "candidate_name", "sdp_votes"]
std = std.drop_duplicates(subset=[c for c in dedupe_cols if c in std.columns]).copy()

print("Standardised SDP rows:", len(std))
print("Rows by source mode:")
display(std["source_mode"].value_counts(dropna=False).reset_index(name="rows"))
print("Rows by election year:")
display(std["election_year"].value_counts(dropna=False).sort_index().reset_index(name="rows"))

display(std[ [c for c in ["election_year", "council_name", "ward_name", "candidate_name", "party_label", "sdp_votes", "valid_votes", "sdp_vote_share_effective", "WD25CD", "ward_code", "result_area_key", "source_mode"] if c in std.columns] ].head(20))


Standardised SDP rows: 219
Rows by source mode:


,source_mode,rows
0,existing_sdp_profile_v1,171
1,provisional_2026_sdp_file,48


Rows by election year:


,election_year,rows
0,2021,68
1,2022,28
2,2023,36
3,2024,28
4,2025,11
5,2026,48


,election_year,council_name,ward_name,candidate_name,party_label,sdp_votes,valid_votes,sdp_vote_share_effective,WD25CD,ward_code,result_area_key,source_mode
0,2021,"Bristol, City Of",Frome Vale,Trueman T.,SDP,112.0,3835.0,0.029205,E05010899,E05010899,2021|CODE|BRISTOL_CITY_OF|E05010899|FROME_VALE,existing_sdp_profile_v1
1,2021,Reading,Caversham,Skelton D.J.A.,SDP,17.0,3112.0,0.005463,E05002321,E05002321,2021|CODE|READING|E05002321|CAVERSHAM,existing_sdp_profile_v1
2,2021,Buckinghamshire,Stone And Waddesdon,Tinay P.D.,SDP,37.0,4005.0,0.009238,E05013159,E05013159,2021|CODE|BUCKINGHAMSHIRE|E05013159|STONE_AND_...,existing_sdp_profile_v1
3,2021,Hartlepool,Burn Valley,Humphries L.P.,SDP,225.0,3218.0,0.069919,E05013038,E05013038,2021|CODE|HARTLEPOOL|E05013038|BURN_VALLEY,existing_sdp_profile_v1
4,2021,Wealden,Hailsham Market,Gander S.R.,SDP,103.0,2484.0,0.041465,E58000368,E58000368,2021|CODE|WEALDEN|E58000368|HAILSHAM_MARKET,existing_sdp_profile_v1
5,2021,Gloucester,Abbeymead,Hameed S.,SDP,32.0,1746.0,0.018328,E05010951,E05010951,2021|CODE|GLOUCESTER|E05010951|ABBEYMEAD,existing_sdp_profile_v1
6,2021,Manchester,Didsbury West,Andrew W.M.,SDP,29.0,5623.0,0.005157,E05011363,E05011363,2021|CODE|MANCHESTER|E05011363|DIDSBURY_WEST,existing_sdp_profile_v1
7,2021,Rochdale,East Middleton,Mudd R.,SDP,71.0,2484.0,0.028583,E05000743,E05000743,2021|CODE|ROCHDALE|E05000743|EAST_MIDDLETON,existing_sdp_profile_v1
8,2021,North Hertfordshire,Letchworth South West,McGetrick M.,SDP,39.0,2758.0,0.014141,E05004779,E05004779,2021|CODE|NORTH_HERTFORDSHIRE|E05004779|LETCHW...,existing_sdp_profile_v1
9,2021,Kingston Upon Hull,Drypool,Waterston J.W.,SDP,8.0,2674.0,0.002992,E05011531,E05011531,2021|CODE|KINGSTON_UPON_HULL|E05011531|DRYPOOL,existing_sdp_profile_v1


## 24b2.4 Patch source ward codes from matched ward summary, if available

This helps 2023 rows that were originally name-only but were fixed by Notebook 16.

In [6]:
ward_summary, ward_summary_path = read_csv_optional(WARD_SUMMARY_MATCHED_FILENAME)
if ward_summary is not None and "result_area_key" in ward_summary.columns:
    ws = ward_summary.copy()
    rename_map = {}
    for source_col, target_col in [
        ("source_geography_code", "patched_source_geography_code"),
        ("source_geography_name", "patched_source_geography_name"),
        ("ward_code", "patched_ward_code"),
        ("lad_code", "patched_lad_code"),
        ("source_boundary_year", "patched_source_boundary_year"),
    ]:
        if source_col in ws.columns:
            rename_map[source_col] = target_col
    ws = ws[["result_area_key"] + list(rename_map.keys())].rename(columns=rename_map).drop_duplicates("result_area_key")
    std = std.merge(ws, on="result_area_key", how="left", validate="many_to_one")

    # Use patched source geography if ward_code is missing.
    if "patched_source_geography_code" in std.columns:
        std["source_geography_code_for_match"] = std["ward_code"].combine_first(std["patched_source_geography_code"].map(norm_code))
    else:
        std["source_geography_code_for_match"] = std["ward_code"]
    if "patched_source_boundary_year" in std.columns:
        std["source_boundary_year_for_match"] = std["boundary_year"].fillna(std["patched_source_boundary_year"])
    else:
        std["source_boundary_year_for_match"] = std["boundary_year"]
else:
    std["source_geography_code_for_match"] = std["ward_code"]
    std["source_boundary_year_for_match"] = std["boundary_year"]

std["source_geography_code_for_match"] = std["source_geography_code_for_match"].map(norm_code)
std["source_boundary_year_for_match"] = to_num(std["source_boundary_year_for_match"]).astype("Int64")

display(std[["election_year", "council_name", "ward_name", "WD25CD", "ward_code", "source_geography_code_for_match", "source_boundary_year_for_match"]].head(20))

Loaded ward_result_summary_v2_with_2023_matches.csv: (15609, 68) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\ward_result_summary_v2_with_2023_matches.csv


,election_year,council_name,ward_name,WD25CD,ward_code,source_geography_code_for_match,source_boundary_year_for_match
0,2021,"Bristol, City Of",Frome Vale,E05010899,E05010899,E05010899,2021
1,2021,Reading,Caversham,E05002321,E05002321,E05002321,2021
2,2021,Buckinghamshire,Stone And Waddesdon,E05013159,E05013159,E05013159,2021
3,2021,Hartlepool,Burn Valley,E05013038,E05013038,E05013038,2021
4,2021,Wealden,Hailsham Market,E58000368,E58000368,E58000368,2021
5,2021,Gloucester,Abbeymead,E05010951,E05010951,E05010951,2021
6,2021,Manchester,Didsbury West,E05011363,E05011363,E05011363,2021
7,2021,Rochdale,East Middleton,E05000743,E05000743,E05000743,2021
8,2021,North Hertfordshire,Letchworth South West,E05004779,E05004779,E05004779,2021
9,2021,Kingston Upon Hull,Drypool,E05011531,E05011531,E05011531,2021


## 24b2.5 Build source-year ward to WD25 crosswalks, if lookup files exist

This is especially important for 2023 rows that were name-only in the raw House of Commons Library workbook.

In [7]:

def find_oa_to_ward_lookup_for_year(year):
    """Find an OA21→WDYY lookup, not a WD→LAD/CED lookup.

    This deliberately rejects files such as wd25_to_lad25_cty25_ced25_eng.csv,
    because those do not contain OA21CD and cannot be used to build an OA-based
    source-year → WD25 crosswalk.
    """
    yy = str(year)[-2:]
    patterns = [
        f"oa21_to_wd{yy}*.csv",
        f"*oa21*wd{yy}*.csv",
        f"*oa21*ward*{yy}*.csv",
    ]

    candidates = []
    for pat in patterns:
        candidates.extend(GEOGRAPHY_DIR.glob(pat))

    valid = []
    for path in sorted(set(candidates), key=lambda p: len(p.name)):
        name = path.name.lower()

        # Reject ward-to-LAD/CED lookups. These are useful elsewhere, but not here.
        if name.startswith(f"wd{yy}_to") or "_to_lad" in name or "_to_cty" in name or "_to_ced" in name:
            continue

        try:
            preview = pd.read_csv(path, nrows=0)
            cols_upper = {c.upper() for c in preview.columns}
            if "OA21CD" in cols_upper and f"WD{yy}CD" in cols_upper:
                valid.append(path)
        except Exception:
            continue

    return valid[0] if valid else None


def standardise_oa_ward_lookup(path, year):
    df = pd.read_csv(path, low_memory=False)
    yy = str(year)[-2:]

    oa_col = next((c for c in df.columns if c.upper() == "OA21CD"), None)
    wd_col = next((c for c in df.columns if c.upper() == f"WD{yy}CD"), None)
    wd_nm = next((c for c in df.columns if c.upper() == f"WD{yy}NM"), None)
    lad_col = next((c for c in df.columns if c.upper() == f"LAD{yy}CD"), None)
    lad_nm = next((c for c in df.columns if c.upper() == f"LAD{yy}NM"), None)

    if oa_col is None or wd_col is None:
        raise ValueError(
            f"Could not identify OA21CD and WD{yy}CD columns in {path}. "
            f"Available columns: {df.columns.tolist()}"
        )

    keep = [oa_col, wd_col]
    if wd_nm: keep.append(wd_nm)
    if lad_col: keep.append(lad_col)
    if lad_nm: keep.append(lad_nm)

    out = df[keep].copy()
    rename = {oa_col: "OA21CD", wd_col: "source_ward_code"}
    if wd_nm: rename[wd_nm] = "source_ward_name"
    if lad_col: rename[lad_col] = "source_lad_code"
    if lad_nm: rename[lad_nm] = "source_lad_name"
    out = out.rename(columns=rename)
    out["source_boundary_year"] = int(year)
    return out.drop_duplicates()


# Need true OA21→WD25 lookup, not wd25_to_lad25_cty25_ced25_eng.csv.
wd25_path = find_oa_to_ward_lookup_for_year(2025)
print("OA21→WD25 lookup selected:", wd25_path)

source_to_wd25_crosswalks = []
if wd25_path is None:
    print("No valid OA21→WD25 lookup found. Source-year crosswalks cannot be built.")
    print("This is not fatal: rows with existing WD25CD can still match directly.")
else:
    wd25 = standardise_oa_ward_lookup(wd25_path, 2025).rename(
        columns={"source_ward_code": "WD25CD", "source_ward_name": "WD25NM"}
    )
    wd25 = wd25[["OA21CD", "WD25CD"]].drop_duplicates()

    for yr in sorted(std["source_boundary_year_for_match"].dropna().astype(int).unique()):
        if yr == 2025:
            continue

        src_path = find_oa_to_ward_lookup_for_year(yr)
        if src_path is None:
            print(f"No valid OA21→WD{str(yr)[-2:]} lookup found for {yr}.")
            continue

        try:
            src = standardise_oa_ward_lookup(src_path, yr)
            cw = src[["OA21CD", "source_ward_code"]].merge(wd25, on="OA21CD", how="inner")

            counts = cw.groupby(["source_ward_code", "WD25CD"]).size().reset_index(name="oa_count")
            totals = counts.groupby("source_ward_code")["oa_count"].transform("sum")
            counts["wd25_match_share"] = counts["oa_count"] / totals

            best = (
                counts.sort_values(["source_ward_code", "wd25_match_share", "oa_count"], ascending=[True, False, False])
                .drop_duplicates("source_ward_code")
            )
            best["source_boundary_year"] = yr
            source_to_wd25_crosswalks.append(best)
            print(f"Built source→WD25 crosswalk for {yr}: {len(best)} source wards from {src_path.name}")
        except Exception as e:
            print(f"Failed source-year crosswalk for {yr}:", e)

if source_to_wd25_crosswalks:
    source_to_wd25 = pd.concat(source_to_wd25_crosswalks, ignore_index=True)
else:
    source_to_wd25 = pd.DataFrame(columns=["source_boundary_year", "source_ward_code", "WD25CD", "wd25_match_share"])

print("Crosswalk rows:", len(source_to_wd25))


OA21→WD25 lookup selected: c:\Users\keena\Documents\Electoral_Tribes\data\geography\oa21_to_wd25_lad25_eng_wal(may25).csv
No valid OA21→WD21 lookup found for 2021.
Built source→WD25 crosswalk for 2022: 7638 source wards from oa21_to_wd22_lad22_ctyua22_rgn22_ctry22_eng_wal.csv
Built source→WD25 crosswalk for 2023: 7608 source wards from oa21_to_wd23_lad23_eng_wal.csv
Built source→WD25 crosswalk for 2024: 7563 source wards from OA21_WD24_LAD24_EW_LU_v2.csv
No valid OA21→WD26 lookup found for 2026.
Crosswalk rows: 22809


## 24b2.6 Match SDP rows to model

In [8]:

# ============================================================
# 2.6 Match SDP rows to the current WD25 model atlas
# ============================================================
# Pandas 3 / newer nullable dtypes can be strict about assigning
# string ward codes into columns that were initially created as
# float NaN columns. Therefore every match/code column is explicitly
# created as string/object-compatible before assignment.
# ============================================================

match = std.copy()

# Defensive name-cleaning columns for the LAD+ward-name fallback.
# This cell may be run after older standardisation cells, or after a partial rerun,
# so do not assume clean_council_name / clean_ward_name already exist.
def _coalesce_existing(df, candidates):
    out = pd.Series(pd.NA, index=df.index, dtype="string")
    for col in candidates:
        if col in df.columns:
            out = out.combine_first(df[col].astype("string"))
    return out

if "clean_council_name" not in match.columns:
    _council_src = _coalesce_existing(
        match,
        ["council_name", "LAD25NM", "LAD26NM", "local_authority", "Authority", "Local Authority"]
    )
    match["clean_council_name"] = _council_src.map(clean_text)
else:
    match["clean_council_name"] = match["clean_council_name"].map(clean_text)

if "clean_ward_name" not in match.columns:
    _ward_src = _coalesce_existing(
        match,
        ["ward_name", "WD25NM", "WD26NM", "division_name", "Ward", "Division"]
    )
    match["clean_ward_name"] = _ward_src.map(clean_text)
else:
    match["clean_ward_name"] = match["clean_ward_name"].map(clean_text)


# Normalise code columns as strings.
for col in ["WD25CD", "source_geography_code_for_match", "source_boundary_year_for_match"]:
    if col in match.columns:
        match[col] = match[col].astype("string").str.strip()

# Ensure model codes are strings.
model_codes = set(pd.Series(list(model_codes), dtype="string").dropna().astype(str))

# Explicit dtypes to avoid LossySetitemError when assigning E-codes.
match["match_method_v2"] = pd.Series("unmatched", index=match.index, dtype="string")
match["match_confidence_v2"] = pd.Series("none", index=match.index, dtype="string")
match["WD25CD_final"] = pd.Series(pd.NA, index=match.index, dtype="string")
match["wd25_match_share_v2"] = np.nan

# ------------------------------------------------------------
# 1. Existing WD25CD direct match.
# ------------------------------------------------------------
direct_mask = match["WD25CD"].astype("string").isin(model_codes)

match.loc[direct_mask, "WD25CD_final"] = match.loc[direct_mask, "WD25CD"].astype("string").values
match.loc[direct_mask, "match_method_v2"] = "existing_wd25cd"
match.loc[direct_mask, "match_confidence_v2"] = "high"

# ------------------------------------------------------------
# 2. Source-year crosswalk match.
# ------------------------------------------------------------
if len(source_to_wd25) > 0:
    sx = source_to_wd25.copy()

    # Merge keys must have identical dtypes on both sides. Use pandas string
    # dtype for both year and ward-code keys to avoid string/int64 merge errors.
    for col in ["source_boundary_year", "source_ward_code", "WD25CD"]:
        if col in sx.columns:
            sx[col] = sx[col].astype("string").str.strip()

    for col in ["source_boundary_year_for_match", "source_geography_code_for_match"]:
        if col in match.columns:
            match[col] = match[col].astype("string").str.strip()

    sx = sx.rename(columns={"WD25CD": "crosswalk_WD25CD"})

    match = match.merge(
        sx[["source_boundary_year", "source_ward_code", "crosswalk_WD25CD", "wd25_match_share"]],
        left_on=["source_boundary_year_for_match", "source_geography_code_for_match"],
        right_on=["source_boundary_year", "source_ward_code"],
        how="left"
    )

    match["crosswalk_WD25CD"] = match["crosswalk_WD25CD"].astype("string").str.strip()
    match["wd25_match_share"] = pd.to_numeric(match["wd25_match_share"], errors="coerce")

    cw_mask = (
        match["WD25CD_final"].isna()
        & match["crosswalk_WD25CD"].isin(model_codes)
        & match["wd25_match_share"].ge(SOURCE_TO_WD25_MIN_SHARE)
    )

    match.loc[cw_mask, "WD25CD_final"] = match.loc[cw_mask, "crosswalk_WD25CD"].astype("string").values
    match.loc[cw_mask, "match_method_v2"] = "source_year_to_wd25_crosswalk"
    match.loc[cw_mask, "match_confidence_v2"] = np.where(
        match.loc[cw_mask, "wd25_match_share"].ge(0.8),
        "high",
        "medium"
    )
    match.loc[cw_mask, "wd25_match_share_v2"] = match.loc[cw_mask, "wd25_match_share"]
else:
    print("No source-year to WD25 crosswalk available. Skipping crosswalk match step.")

# ------------------------------------------------------------
# 3. Conservative exact name fallback within LAD/ward names.
# ------------------------------------------------------------
model_name = model_lookup.copy()
model_name["WD25CD"] = model_name["WD25CD"].astype("string").str.strip()
model_name["clean_lad_name"] = model_name.get("LAD25NM", pd.Series(index=model_name.index)).map(clean_text)
model_name["clean_ward_name"] = model_name.get("WD25NM", pd.Series(index=model_name.index)).map(clean_text)
name_key = model_name[["WD25CD", "clean_lad_name", "clean_ward_name"]].dropna().drop_duplicates()

still = match["WD25CD_final"].isna()

if still.any():
    # Preserve original index through the merge so that matches align back safely.
    tmp = match.loc[still, ["clean_council_name", "clean_ward_name"]].copy()
    tmp["_original_index"] = tmp.index

    name_merge = tmp.merge(
        name_key,
        left_on=["clean_council_name", "clean_ward_name"],
        right_on=["clean_lad_name", "clean_ward_name"],
        how="left"
    )

    name_merge = name_merge[name_merge["WD25CD"].astype("string").isin(model_codes)].copy()
    name_merge = name_merge.drop_duplicates("_original_index")

    if len(name_merge) > 0:
        idx = name_merge["_original_index"].values
        vals = name_merge["WD25CD"].astype("string").values
        match.loc[idx, "WD25CD_final"] = vals
        match.loc[idx, "match_method_v2"] = "exact_lad_ward_name_fallback"
        match.loc[idx, "match_confidence_v2"] = "medium"

# ------------------------------------------------------------
# 4. Join model fields.
# ------------------------------------------------------------
model_lookup = model_lookup.copy()
model_lookup["WD25CD"] = model_lookup["WD25CD"].astype("string").str.strip()

model_cols = [c for c in [
    "WD25CD", "WD25NM", "LAD25CD", "LAD25NM", "analysis_region", "strategic_lane", "revised_strategic_lane_v2",
    "report_confidence_band_v2", "report_caveat_summary_v2", "initial_watchlist_score", "demographic_relevance_score",
    "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score", "dominant_cluster_name", "second_cluster_name",
    "latest_election_top_party_bucket", "latest_election_runner_up_party_bucket"
] if c in model_lookup.columns]

profile = match.merge(
    model_lookup[model_cols],
    left_on="WD25CD_final",
    right_on="WD25CD",
    how="left",
    suffixes=("", "_model")
)

profile["matched_to_model_v2"] = profile["WD25CD_model"].notna() if "WD25CD_model" in profile.columns else profile["WD25CD_final"].isin(model_codes)
profile["WD25CD"] = profile["WD25CD_final"].combine_first(profile.get("WD25CD", pd.Series(index=profile.index, dtype="string")))

print("Match methods:")
display(profile["match_method_v2"].value_counts(dropna=False).reset_index(name="rows"))
print("Matched to model:", int(profile["matched_to_model_v2"].sum()), "of", len(profile))

preview_cols = [c for c in [
    "election_year", "council_name", "ward_name", "candidate_name", "sdp_votes", "valid_votes", "sdp_vote_share_effective",
    "WD25CD", "match_method_v2", "match_confidence_v2", "matched_to_model_v2", "dominant_cluster_name"
] if c in profile.columns]

display(profile[preview_cols].head(30))


Match methods:


,match_method_v2,rows
0,existing_wd25cd,141
1,source_year_to_wd25_crosswalk,37
2,unmatched,23
3,exact_lad_ward_name_fallback,18


Matched to model: 196 of 219


,election_year,council_name,ward_name,candidate_name,sdp_votes,valid_votes,sdp_vote_share_effective,WD25CD,match_method_v2,match_confidence_v2,matched_to_model_v2,dominant_cluster_name
0,2021,"Bristol, City Of",Frome Vale,Trueman T.,112.0,3835.0,0.029205,E05010899,existing_wd25cd,high,True,Settled Working Families / Skilled Trades Suburbs
1,2021,Reading,Caversham,Skelton D.J.A.,17.0,3112.0,0.005463,E05013866,exact_lad_ward_name_fallback,medium,True,Stable Suburban Professionals
2,2021,Buckinghamshire,Stone And Waddesdon,Tinay P.D.,37.0,4005.0,0.009238,E05013159,unmatched,none,False,NaN
3,2021,Hartlepool,Burn Valley,Humphries L.P.,225.0,3218.0,0.069919,E05013038,existing_wd25cd,high,True,Post-Industrial Estates / Deprived Working Com...
4,2021,Wealden,Hailsham Market,Gander S.R.,103.0,2484.0,0.041465,E58000368,unmatched,none,False,NaN
5,2021,Gloucester,Abbeymead,Hameed S.,32.0,1746.0,0.018328,E05010951,existing_wd25cd,high,True,Settled Working Families / Skilled Trades Suburbs
6,2021,Manchester,Didsbury West,Andrew W.M.,29.0,5623.0,0.005157,E05011363,existing_wd25cd,high,True,Cosmopolitan Young Professional Core
7,2021,Rochdale,East Middleton,Mudd R.,71.0,2484.0,0.028583,E05014037,exact_lad_ward_name_fallback,medium,True,Settled Working Families / Skilled Trades Suburbs
8,2021,North Hertfordshire,Letchworth South West,McGetrick M.,39.0,2758.0,0.014141,E05015772,exact_lad_ward_name_fallback,medium,True,Stable Suburban Professionals
9,2021,Kingston Upon Hull,Drypool,Waterston J.W.,8.0,2674.0,0.002992,E05011531,existing_wd25cd,high,True,Post-Industrial Estates / Deprived Working Com...


## 24b2.7 Outputs and summaries

In [9]:
EXPECTED_COUNTS = {2021: 68, 2022: 30, 2023: 36, 2024: 28, 2025: 11, 2026: 48}
observed = profile.groupby("election_year").size().rename("observed_sdp_candidate_rows").reset_index()
count_check = pd.DataFrame({"election_year": list(EXPECTED_COUNTS.keys()), "expected_user_provided": list(EXPECTED_COUNTS.values())})
count_check = count_check.merge(observed, on="election_year", how="left").fillna({"observed_sdp_candidate_rows": 0})
count_check["observed_sdp_candidate_rows"] = count_check["observed_sdp_candidate_rows"].astype(int)
count_check["difference_observed_minus_expected"] = count_check["observed_sdp_candidate_rows"] - count_check["expected_user_provided"]

def summary_group(cols):
    g = profile.groupby(cols, dropna=False).agg(
        sdp_candidate_rows=("candidate_name", "count"),
        matched_rows=("matched_to_model_v2", "sum"),
        total_sdp_votes=("sdp_votes", "sum"),
        mean_sdp_vote_share=("sdp_vote_share_effective", "mean"),
        median_sdp_vote_share=("sdp_vote_share_effective", "median"),
        max_sdp_vote_share=("sdp_vote_share_effective", "max"),
        mean_model_score=("initial_watchlist_score", "mean"),
    ).reset_index()
    return g.sort_values(["total_sdp_votes", "sdp_candidate_rows"], ascending=[False, False])

by_year = summary_group(["election_year"])
by_region = summary_group(["analysis_region"])
by_tribe = summary_group(["dominant_cluster_name"])
by_party = summary_group(["latest_election_top_party_bucket"])
repeat = summary_group(["WD25CD", "LAD25NM", "WD25NM"])
repeat = repeat[repeat["sdp_candidate_rows"].ge(1)].sort_values(["sdp_candidate_rows", "max_sdp_vote_share", "total_sdp_votes"], ascending=[False, False, False])
highest = profile.sort_values(["sdp_vote_share_effective", "sdp_votes"], ascending=[False, False])
unmatched = profile[~profile["matched_to_model_v2"]].copy()

# Correlation only if enough non-null observations.
corr_cols = ["sdp_vote_share_effective", "initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score"]
corr_data = profile[corr_cols].dropna()
if len(corr_data) >= 3:
    corr = corr_data.corr().reset_index().rename(columns={"index": "metric"})
else:
    corr = pd.DataFrame(columns=["metric"] + corr_cols)

# Save v2 filenames expected downstream.
profile.to_csv(OUTPUT_DIR / "sdp_campaign_wards_profile_v2.csv", index=False)
count_check.to_csv(OUTPUT_DIR / "sdp_candidate_count_check_v2.csv", index=False)
by_year.to_csv(OUTPUT_DIR / "sdp_performance_by_year_v2.csv", index=False)
by_region.to_csv(OUTPUT_DIR / "sdp_performance_by_region_v2.csv", index=False)
by_tribe.to_csv(OUTPUT_DIR / "sdp_performance_by_dominant_tribe_v2.csv", index=False)
by_party.to_csv(OUTPUT_DIR / "sdp_performance_by_latest_top_party_v2.csv", index=False)
repeat.to_csv(OUTPUT_DIR / "sdp_repeat_campaign_wards_v2.csv", index=False)
highest.to_csv(OUTPUT_DIR / "sdp_highest_vote_share_cases_v2.csv", index=False)
unmatched.to_csv(OUTPUT_DIR / "sdp_unmatched_results_review_v2.csv", index=False)
corr.to_csv(OUTPUT_DIR / "sdp_campaign_vs_model_score_correlation_v2.csv", index=False)

print("Candidate count check")
display(count_check)
print("Matched rows:", int(profile["matched_to_model_v2"].sum()), "Unmatched:", len(unmatched))
print("Saved outputs to", OUTPUT_DIR)

Candidate count check


,election_year,expected_user_provided,observed_sdp_candidate_rows,difference_observed_minus_expected
0,2021,68,68,0
1,2022,30,28,-2
2,2023,36,36,0
3,2024,28,28,0
4,2025,11,11,0
5,2026,48,48,0


Matched rows: 196 Unmatched: 23
Saved outputs to c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v2
